# Without History

In [ ]:
import re
from typing import Optional, TypedDict, List
from langgraph.graph import StateGraph, START, END
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
import dotenv
import json

dotenv.load_dotenv()

def clean_json_response(response: str) -> str:
    # print(response)
    match = re.search(r"```json(.*?)```", response.strip(), re.DOTALL)

    return match.group(1).strip()

# Define the shipping workflow state.
class ShippingWorkflowState(TypedDict):
    full_name: Optional[str]
    company_name: Optional[str]
    company_address: Optional[str]
    phone_number: Optional[str]
    email: Optional[str]
    number_of_containers: Optional[str]

chat_history = []
global current_question
current_question=None

# List of required fields (in order)
REQUIRED_FIELDS = [
    "full_name", "company_name", "company_address", "phone_number", "email",
    "number_of_containers"
]

def is_complete(state: ShippingWorkflowState) -> bool:
    return all(state.get(field) for field in REQUIRED_FIELDS)

# Initialize the LLM.
llm = ChatOpenAI(model="gpt-4o")

# Prompt template to ask the user for a specific piece of information.
question_prompt = PromptTemplate(
    input_variables=["history", "current_state"],
    template=(
        # Chat History: {history}
        """
        This is my current data from the user:
        {current_state}
        user gave me this further detail. You have to update the state accordingly.  There could be potentially misreadings in the data, you have to carefully look a the new data and update the state accordingly.
        You have to ask the user for the information that is missing in the current state. Its better to ask the user in order of the fields in the current state.
        Here is the state description: 
        state description
        [
            full_name: [first name , last name (optional)]
            company_name: [Company name where the user works at ]
            company_address: [Company address of the user's company]
            phone_number: [Phone number of the user]
            email: [Email address of the user]
            number_of_containers: [Number of containers the user wants to ship]
        ]
        Your response should only contain the question you want to ask the user.
        """
    )
)

# state_update_prompt = PromptTemplate(
#     input_variables=["history", "current_state","question","response"],
#     template=(
#         # Chat History: {history}
#         """
#         This is my current data from the user:
#         {current_state}
#         user gave me this further detail. You have to update the state accordingly.  There could be potentially misreadings in the data, you have to carefully look a the new data and update the state accordingly.
#         You have asked the user for the information that is missing in the current state and the user has provided the information. You have to update the state accordingly.
#         The user may give you some information that is already in the state but is not correct. You have to update the state with the new information. 
#         Output only a json of the new state after updating the state with the new information.
#         Here is the state description: 
#         state description
#         [
#             full_name: [first name , last name (optional)]
#             company_name: [Company name where the user works at ]
#             company_address: [Company address of the user's company]
#             phone_number: [Phone number of the user]
#             email: [Email address of the user]
#             number_of_containers: [Number of containers the user wants to ship]
#         ]
#         question: {question}
#         Response: {response}
#         """
#     )
# )

state_update_prompt = PromptTemplate(
    input_variables=["history", "current_state", "question", "response"],
    template=(
        """
        This is my current data from the user:
        {current_state}

        The user provided the following response to the question:
        Question: {question}
        Response: {response}

        Your tasks are:
        1. Update the state with the new information provided by the user. If the user provides information that conflicts with the existing state, prioritize the new information.
        2. If the user asked a question or needs clarification, provide a friendly and helpful response.

        State Description:
        [
            full_name: [first name, last name (optional)]
            company_name: [Company name where the user works at]
            company_address: [Company address of the user's company]
            phone_number: [Phone number of the user]
            email: [Email address of the user]
            number_of_containers: [Number of containers the user wants to ship]
        ]

        Output Format:
        {{
            "updated_state": {{
                "full_name": "...",
                "company_name": "...",
                "company_address": "...",
                "phone_number": "...",
                "email": "...",
                "number_of_containers": "..."
            }},
            "response": "Your response to the user here."
        }}

        Output only the JSON object as described above.
        """
    )
)


# --- Node Functions ---

def ask_question_node(state: ShippingWorkflowState) -> ShippingWorkflowState:
    # Check if the state is complete.
    if is_complete(state):
        return state
    # print("in ask_question_node")
    # use the question prompt to ask the user for the next piece of information

    # Convert chat_history list to a suitable string representation
    chat_history_str = "\n".join(str(msg) for msg in chat_history) if chat_history else ""
    # print("chat_history_str: ",chat_history_str)
    prompt_his = question_prompt.format(history=chat_history_str, current_state=state)
    # print("prompt:",prompt)
    response=LLMChain(llm=llm, prompt=question_prompt).run({
                                                            # "history": chat_history_str, 
                                                            "current_state": state})
    print("response question",response)
    # cleaned_res=clean_json_response(response)
    current_question=response
    chat_history.append(prompt_his.join("Question: "+response))

    return state

def process_answer_node(state: ShippingWorkflowState) -> ShippingWorkflowState:
    # Use the state_update_prompt to update the state with the user's answer.
    user_input = input(current_question)

    # Convert chat_history list to a suitable string representation
    chat_history_str = "\n".join(str(msg) for msg in chat_history) if chat_history else ""
    prompt_his = state_update_prompt.format(history=chat_history_str, current_state=state,question=current_question,response=user_input)
    # print("prompt",prompt)
    response=LLMChain(llm=llm, prompt=state_update_prompt).run({
                                                    # "history": chat_history_str,
                                                    "current_state": state,
                                                    "question": current_question,
                                                    "response": user_input
                                                })
    # print("response json",response)
    cleaned_res=clean_json_response(response)
    
    try:
        new_json = json.loads(cleaned_res)  # Use json.loads instead of eval
        new_state = new_json["updated_state"]
        print("Response from Chatbot: ",new_json["response"])
        state.update(new_state)
        print("Updated State: ", state)
    except json.JSONDecodeError as e:
        print("Error in decoding the response json: ", e)
        print("Response: ", cleaned_res)
    chat_history.append(prompt_his.join("Updated State: "+cleaned_res))
    
    
    return state

def workflow_complete_node(state: ShippingWorkflowState) -> ShippingWorkflowState:
    # Check if the state is complete.
    if is_complete(state):
        return state
    return state

# --- Conditional Routing Functions ---
def route_after_ask(state: ShippingWorkflowState) -> str:
    return "workflow_complete" if is_complete(state) else "process_answer"

def route_after_process(state: ShippingWorkflowState) -> str:
    return "workflow_complete" if is_complete(state) else "ask_question"

# --- Build the LangGraph State Graph ---
workflow = StateGraph(ShippingWorkflowState)

workflow.add_node("ask_question", ask_question_node)
workflow.add_node("process_answer", process_answer_node)
workflow.add_node("workflow_complete", workflow_complete_node)

workflow.add_edge(START, "ask_question")

workflow.add_conditional_edges(
    "ask_question",
    route_after_ask,
    {"workflow_complete": "workflow_complete", "process_answer": "process_answer"}
)

workflow.add_conditional_edges(
    "process_answer",
    route_after_process,
    {"workflow_complete": "workflow_complete", "ask_question": "ask_question"}
)

workflow.add_edge("workflow_complete", END)

app = workflow.compile()

# --- Main Loop ---
if __name__ == "__main__":
    # Initialize state with all fields empty and an empty chat history.
    state: ShippingWorkflowState = {
        "full_name": None,
        "company_name": None,
        "company_address": None,
        "phone_number": None,
        "email": None,
        "number_of_containers": None
    }
print("state",state)
state = app.invoke(state)



state {'full_name': None, 'company_name': None, 'company_address': None, 'phone_number': None, 'email': None, 'number_of_containers': None}
response question Could you please provide your full name?
Response from Chatbot:  Thank you, Mustafa! If you have any more information to provide or if you have any questions, please feel free to let me know. I'm here to help.
Updated State:  {'full_name': 'mustafa', 'company_name': None, 'company_address': None, 'phone_number': None, 'email': None, 'number_of_containers': None}
response question Could you please provide the company name where you work?
Response from Chatbot:  Sure, I can help you with that! To fry an egg, start by heating a small amount of oil or butter in a non-stick skillet over medium heat. Crack the egg into the skillet and let it cook until the white is set and the yolk is still runny, or cook to your preferred doneness. You can cover the skillet with a lid if you’d like to help set the top of the egg. Season with salt and p

In [ ]:

test_inputs = [
    "HI I am John Doe from ABC Inc. My address is 123 Main St, Dallas, TX 75201. Phone: 555-123-4567, Email: jhondoe@gmail.com",
    "We have 4 containers to  ship",
    "The container is 40ft and loaded. Pickup address is 456 Elm St, Dallas, TX 75202",
    "The delivery address is 456 Elm St, Dallas, TX 75202"
]

# With History

In [ ]:
import re
from typing import Optional, TypedDict, List
from langgraph.graph import StateGraph, START, END
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
import dotenv
import json

dotenv.load_dotenv()

def clean_json_response(response: str) -> str:
    # Extract JSON between ```json ... ```
    match = re.search(r"```json(.*?)```", response.strip(), re.DOTALL)
    return match.group(1).strip()

# Define the shipping workflow state.
class ShippingWorkflowState(TypedDict):
    full_name: Optional[str]
    company_name: Optional[str]
    company_address: Optional[str]
    phone_number: Optional[str]
    email: Optional[str]
    number_of_containers: Optional[str]
    empty_or_loaded: Optional[List[str]]  
    pickup_address: Optional[List[str]]   
    delivery_address: Optional[List[str]]

# --- Chat History Store ---
# A global dictionary to store chat history per session.
store = {}

def get_by_session_id(session_id: str) -> List[str]:
    if session_id not in store:
        store[session_id] = []
    return store[session_id]

# For this example, we use a fixed session id.
session_id = "foo"

# List of required fields (in order)
REQUIRED_FIELDS = [
    "full_name", "company_name", "company_address", "phone_number", "email",
    "number_of_containers"
]

def is_complete(state: ShippingWorkflowState) -> bool:
    req =all(state.get(field) for field in REQUIRED_FIELDS)
    if req:
        if state.get("number_of_containers") and int(state.get("number_of_containers")) == len(state.get("empty_or_loaded")):
            return False
    return req

# Initialize the LLM.
llm = ChatOpenAI(model="gpt-4o")

# Prompt template to ask the user for missing information.
question_prompt = PromptTemplate(
    input_variables=["history", "current_state"],
    template=(
        """
        History:
        {history}
        This is my current data from the user:
        {current_state}
        1. The user has provided some details. Update the state if needed.
        2. Please ask for the missing information in the order of the fields.
        3. Keep the history of the chat in mind when asking the next question.
        4. The conversational can deviate from the expected flow. Be prepared to handle that.
        5. After a few interactions if the conversation is deviating, try to bring it back on track but in a way that feels natural.
        
        State description:
        [
            full_name: [first name, last name (optional)]
            company_name: [Company name where the user works]
            company_address: [Company address of the user's company]
            phone_number: [Phone number of the user]
            email: [Email address of the user]
            number_of_containers: [Number of containers the user wants to ship]
                        empty_or_loaded: [Empty or loaded status of the containers. If number_of_containers = 3, this will be a list of 3 elements]
            pickup_address: [Pickup address for the containers. If number_of_containers = 3, this will be a list of 3 elements]
            delivery_address: [Delivery address for the containers. If number_of_containers = 3, this will be a list of 3 elements]
        ]
        
        Your response should only contain the question you want to ask.
        """
    )
)

# Prompt template to update the state based on the user's response.
state_update_prompt = PromptTemplate(
    input_variables=["history", "current_state", "question", "response"],
    template=(
        """
        History:
        {history}
        This is my current data from the user:
        {current_state}

        The user provided the following response to the question:
        Question: {question}
        Response: {response}

        Your tasks are:
        1. Update the state with the new information provided. If there is conflicting info, prioritize the new input.
        2. If the user asked a question or needs clarification, provide a friendly answer.
        3. User may ask some questions which have their answers in the history. You can use the history to answer those questions.

        State description:
        [
            full_name: [first name, last name (optional)]
            company_name: [Company name where the user works]
            company_address: [Company address of the user's company]
            phone_number: [Phone number of the user]
            email: [Email address of the user]
            number_of_containers: [Number of containers the user wants to ship]
            empty_or_loaded: [Empty or loaded status of the containers. For example if number_of_containers = 3, this will be a list of 3 elements]
            pickup_address: [Pickup address for the containers. For example if number_of_containers = 3, this will be a list of 3 elements]
            delivery_address: [Delivery address for the containers. For example if number_of_containers = 3, this will be a list of 3 elements]
        ]

        Output Format:
        {{
            "updated_state": {{
                "full_name": "...",
                "company_name": "...",
                "company_address": "...",
                "phone_number": "...",
                "email": "...",
                "number_of_containers": "...",
                "empty_or_loaded": ["...", "...", "..."],
                "pickup_address": ["...", "...", "..."],
                "delivery_address": ["...", "...", "..."]
            }},
            "response": "Your response to the user here."
        }}

        Output only the JSON object.
        """
    )
)

# Global variable to hold the current question.
current_question = None

# --- Node Functions ---

def ask_question_node(state: ShippingWorkflowState) -> ShippingWorkflowState:
    if is_complete(state):
        return state

    # Retrieve session history and format as string.
    history = get_by_session_id(session_id)
    history_str = "\n".join(history)
    
    # Format the prompt with the current state and chat history.
    formatted_prompt = question_prompt.format(history=history_str, current_state=state)
    
    # Call the LLM to generate a question.
    response = LLMChain(llm=llm, prompt=question_prompt).run({
        "history": history_str,
        "current_state": state
    })
    
    global current_question
    current_question = response
    
    # Append the question to the chat history.
    history.append("\nQuestion asked: " + response)
    return state

def process_answer_node(state: ShippingWorkflowState) -> ShippingWorkflowState:
    history = get_by_session_id(session_id)
    history_str = "\n".join(history)
    
    # Ask the user for their input in response to the current question.
    user_input = input(current_question + "\nYour answer: ")
    
    # Format the state update prompt.
    formatted_prompt = state_update_prompt.format(
        history=history_str, current_state=state,
        question=current_question, response=user_input
    )
    
    # Call the LLM to update the state.
    response = LLMChain(llm=llm, prompt=state_update_prompt).run({
        "history": history_str,
        "current_state": state,
        "question": current_question,
        "response": user_input
    })
    
    cleaned_res = clean_json_response(response)
    try:
        new_json = json.loads(cleaned_res)
        new_state = new_json["updated_state"]
        # clear the ouput before printing
        print("\033[H\033[J")
        print("Response from Chatbot: ", new_json["response"])
        
        keys_to_exclude = {'empty_or_loaded', 'pickup_address', 'delivery_address'}
        state.update({k: v for k, v in new_state.items() if k not in keys_to_exclude})
        state.update(new_state)
        print("Updated State: ", state)
    except json.JSONDecodeError as e:
        print("Error decoding JSON:", e)
        print("Response:", cleaned_res)
    
    # Append the update to the chat history.
    history.append("User Input: "+user_input+"Response from Chatbot: "+new_json["response"]+"\nUpdated State: " + cleaned_res)
    return state

def more_than_3_node(state: ShippingWorkflowState) -> ShippingWorkflowState:
    # Check if the state is complete.
    if is_complete(state):
        return state
    return state


def workflow_complete_node(state: ShippingWorkflowState) -> ShippingWorkflowState:
    # Simply return the state when complete.
    return state

# --- Conditional Routing Functions ---
def route_after_ask(state: ShippingWorkflowState) -> str:
    return "workflow_complete" if is_complete(state) else "process_answer"

def route_after_process(state: ShippingWorkflowState) -> str:
    return "workflow_complete" if is_complete(state) else "ask_question"

# --- Build the LangGraph State Graph ---
workflow = StateGraph(ShippingWorkflowState)
workflow.add_node("ask_question", ask_question_node)
workflow.add_node("process_answer", process_answer_node)
workflow.add_node("more_than_3", more_than_3_node)

workflow.add_node("workflow_complete", workflow_complete_node)

workflow.add_edge(START, "ask_question")
workflow.add_conditional_edges(
    "ask_question",
    route_after_ask,
    {"workflow_complete": "workflow_complete", "process_answer": "process_answer"}
)
workflow.add_conditional_edges(
    "process_answer",
    route_after_process,
    {"workflow_complete": "workflow_complete", "ask_question": "ask_question"}
)
workflow.add_edge("workflow_complete", END)

app = workflow.compile()

# --- Main Loop ---
if __name__ == "__main__":
    # Initialize state with all fields empty.
    state: ShippingWorkflowState = {
        "full_name": None,
        "company_name": None,
        "company_address": None,
        "phone_number": None,
        "email": None,
        "number_of_containers": None
    }
    print("Initial state:", state)
    state = app.invoke(state)
    print("Final state:", state)
    print("Chat history store:", store)


Initial state: {'full_name': None, 'company_name': None, 'company_address': None, 'phone_number': None, 'email': None, 'number_of_containers': None, 'empty_or_loaded': None, 'pickup_address': None, 'delivery_address': None, 'budget': None}

Response from Chatbot:  Thank you, John Doe, for providing your information. If you have any more details to share or questions to ask, feel free to let me know!
Updated State:  {'full_name': 'John Doe', 'company_name': 'ABC Inc.', 'company_address': '123 Main St, Dallas, TX 75201', 'phone_number': '555-123-4567', 'email': 'jhondoe@gmail.com', 'number_of_containers': None, 'empty_or_loaded': [None, None, None], 'pickup_address': [None, None, None], 'delivery_address': [None, None, None], 'budget': None}

Response from Chatbot:  Great! We have noted that you want to ship 2 containers. Could you please let us know whether these containers are empty or loaded, and also provide the pickup and delivery addresses for each container?
Updated State:  {'full

KeyboardInterrupt: 

# Final Version State Dictionary 

In [42]:
import re
from typing import Optional, TypedDict, List
from langgraph.graph import StateGraph, START, END
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
import dotenv
import json

dotenv.load_dotenv()

def clean_json_response(response: str) -> str:
    # Extract JSON between ```json ... ```
    match = re.search(r"```json(.*?)```", response.strip(), re.DOTALL)
    return match.group(1).strip()

class ShippingWorkflowState(TypedDict):
    personal_detail: Optional[dict]  
    more_3: Optional[dict] 
    less_3: Optional[dict] 


# --- Chat History Store ---
# A global dictionary to store chat history per session.
store = {}

def get_by_session_id(session_id: str) -> List[str]:
    if session_id not in store:
        store[session_id] = []
    return store[session_id]

# For this example, we use a fixed session id.
session_id = "foo"

def is_complete_personal_detail(state: ShippingWorkflowState) -> bool:
    personal_detail = all(state["personal_detail"].get(field) for field in ["full_name", "company_name", "company_address", "phone_number", "email","number_of_containers"])


    return  personal_detail

def is_complete_more3(state: ShippingWorkflowState) -> bool:
    required_fields = [ "size", "empty_or_loaded", "pickup_address", "delivery_address"]
    
    # Check if all required fields are present
    if not all(state["more_3"].get(field) for field in required_fields):
        return False
    
    # Check if all elements in the lists are present
    for field in ["size", "empty_or_loaded", "pickup_address", "delivery_address"]:
        if any(not item for item in state["more_3"].get(field, [])):
            return False
    
    return True



def is_complete_less3(state: ShippingWorkflowState) -> bool:
    required_fields = [ "used_service_before", "size", "empty_or_loaded", "hazardous", "new_customer", "pickup_address", "lifting_setup", "container_door_opening_pickup", "pickup_surface_type", "pickup_location_grade", "delivery_address", "dropping_setup", "container_door_opening_drop_off","drop_off_surface_type","drop_off_location_grade"]
    
    # Check if all required fields are present
    if not all(state["less_3"].get(field) for field in required_fields):
        return False
    
    # Check if all elements in the lists are present
    for field in ["size", "empty_or_loaded", "hazardous", "new_customer", "pickup_address", "lifting_setup", "container_door_opening_pickup", "pickup_surface_type", "pickup_location_grade", "delivery_address", "dropping_setup", "container_door_opening_drop_off","drop_off_surface_type","drop_off_location_grade"]:
        if any(not item for item in state["less_3"].get(field, [])):
            return False
    
    return True

# Initialize the LLM.
llm = ChatOpenAI(model="gpt-4o")

# Prompt template to ask the user for missing information.
question_prompt_personal_detail = PromptTemplate(
    input_variables=["history", "current_state"],
    template=(
        """
        History:
        {history}
        This is my current data from the user:
        {current_state}
        1. The user has provided some details. Update the state if needed.
        2. Please ask for the missing information in the order of the fields.
        3. Ask in natural conversation flow
        4. Group similar questions when possible. 
        5. Keep the history of the chat in mind when asking the next question.
        6. The conversational can deviate from the expected flow. Be prepared to handle that.
        7. After a few interactions if the conversation is deviating, try to bring it back on track but in a way that feels natural.
        State description:
        [
            full_name: [first name, last name (optional)]
            company_name: [Company name where the user works]
            company_address: [Company address of the user's company]
            phone_number: [Phone number of the user]
            email: [Email address of the user]
            number_of_containers: [Number of containers the user wants to ship]
        ]
        
        Your response should only contain the question you want to ask.
        """
    )
)

# Prompt template to update the state based on the user's response.
state_update_prompt_personal_detail = PromptTemplate(
    input_variables=["history", "current_state", "question", "response"],
    template=(
        """
        History:
        {history}
        This is my current data from the user:
        {current_state}

        The user provided the following response to the question:
        Question: {question}
        Response: {response}

        Your tasks are:
        1. Update the state with the new information provided. If there is conflicting info, prioritize the new input.
        2. If the user asked a question or needs clarification, provide a friendly answer.
        3. User may ask some questions which have their answers in the history. You can use the history to answer those questions.

        State description:
        [
            full_name: [first name, last name (optional)]
            company_name: [Company name where the user works]
            company_address: [Company address of the user's company]
            phone_number: [Phone number of the user]
            email: [Email address of the user]
        ]

        Output Format:
        {{
            "updated_state": {{
                "full_name": "...",
                "company_name": "...",
                "company_address": "...",
                "phone_number": "...",
                "email": "...",
                "number_of_containers": "..."
            }},
            "response": "Your response to the user here."
        }}

        Output only the JSON object.
        """
    )
)


# Prompt template to ask the user for missing information.
question_prompt_more3 = PromptTemplate(
    input_variables=["history", "current_state"],
    template=(
        """
        History:
        {history}
        This is my current data from the user:
        {current_state}
        1. The user has provided some details. Update the state if needed.
        2. Please ask for the missing information in the order of the fields.
        3. Ask in natural conversation flow
        4. Group similar questions when possible. 
        5. Keep the history of the chat in mind when asking the next question.
        6. The conversational can deviate from the expected flow. Be prepared to handle that.
        7. After a few interactions if the conversation is deviating, try to bring it back on track but in a way that feels natural.
        
        State description:
        [
            size: [Size of the containers. If number_of_containers = 3, this will be a list of 3 elements]
            empty_or_loaded: [Empty or loaded status of the containers. If number_of_containers = 3, this will be a list of 3 elements]
            pickup_address: [Pickup address for the containers. If number_of_containers = 3, this will be a list of 3 elements]
            delivery_address: [Delivery address for the containers. If number_of_containers = 3, this will be a list of 3 elements]
        ]
        
        Your response should only contain the question you want to ask.
        """
    )
)

# Prompt template to update the state based on the user's response.
state_update_prompt_more3 = PromptTemplate(
    input_variables=["history", "current_state", "question", "response"],
    template=(
        """
        History:
        {history}
        This is my current data from the user:
        {current_state}

        The user provided the following response to the question:
        Question: {question}
        Response: {response}

        Your tasks are:
        1. Update the state with the new information provided. If there is conflicting info, prioritize the new input.
        2. If the user asked a question or needs clarification, provide a friendly answer.
        3. User may ask some questions which have their answers in the history. You can use the history to answer those questions.

        State description:
        [
            number_of_containers: [Number of containers the user wants to ship]
            size: [Size of the containers. If number_of_containers = 3, this will be a list of 3 elements]
            empty_or_loaded: [Empty or loaded status of the containers. For example if number_of_containers = 3, this will be a list of 3 elements]
            pickup_address: [Pickup address for the containers. For example if number_of_containers = 3, this will be a list of 3 elements]
            delivery_address: [Delivery address for the containers. For example if number_of_containers = 3, this will be a list of 3 elements]
        ]

        Output Format:
        {{
            "updated_state": {{
                "size": ["...", "...", "..."],
                "empty_or_loaded": ["...", "...", "..."],
                "pickup_address": ["...", "...", "..."],
                "delivery_address": ["...", "...", "..."]
            }},
            "response": "Your response to the user here."
        }}

        Output only the JSON object.
        """
    )
)



# Less than 3 Containers - Asking Question
question_prompt_less3 = PromptTemplate(
    input_variables=["history", "current_state"],
    template=(
        """
History:
{history}
Current container details:
{current_state}

We still need more information to complete your request. Please ask for the missing details in the following order.
Ask in natural conversation flow. Group similar questions when possible. Use these guidelines:

State fields:
- used_service_before: Have you used our service before? 
  - If Yes, say: "Before we proceed, I recommend reviewing our requirements page to ensure all containers meet our safety standards for transport. This includes restrictions on overhangs or protrusions and proper corner castings. Do all your containers meet these criteria? If not, I’d be happy to guide you through our requirements. Please choose one of the two options."
  - If No, say: "I recommend reviewing our full requirements page to better understand our services, as we are not your typical container transport company. This should help clarify our offerings. Feel free to ask if you have any questions during the quote process."
- size: Provide the size for each container (list).
- empty_or_loaded: Indicate if each container is empty or loaded (list).
- hazardous: Are you transporting any hazardous or combustible materials (e.g., propane, paint, etc.)? If none, respond "No." If yes, please specify. If yes, then add: "Thank you for letting me know about the propane tanks. I’ll note that. Is there anything else hazardous in your shipment?"
- new_customer: Are you a new customer? 
  - If Yes, ask: "Does your container have the universal 5/8 inch corner castings in good condition without major dents or defects? Thank you – could you provide more details on their condition?"
  - Also ask: "Does your container have any protrusions on the ends or top (e.g., air conditioning units, brackets, metal signage, or electrical boxes)? Thank you – if there’s a protrusion on top, we may have safety concerns, but it's case-specific. If possible, please send photos later via email."
  - Then ask: "Do either of the long sides of your container have any protrusions (like air conditioning units, brackets, metal signage, or electrical boxes)? If yes, please describe them. If not, ask: 'Is there any additional information we should know about the container or its contents?'"
- pickup_address: The pickup address (list).
- lifting_setup: Describe the lifting setup (e.g., right/left side load/unload and options like 20, 40, or 60 feet). If unsure, say: "No problem, I understand. These guidelines are flexible, and we know every setup is unique. Let's continue and we can follow up later if needed. You may also send photos later."
- container_door_opening_pickup: If applicable, ask: "Which way does the container door open? Please choose one: [A - towards the truck cabin, B - Right side, C - behind the truck, D - Left side]. If unsure, feel free to skip."
- pickup_surface_type: What is the type of surface at the pickup location? (e.g., concrete, asphalt, grass, or dirt)
- pickup_location_grade: What is the approximate grade of the pickup location? If unsure, choose one: Flat Surface, Mild incline, or Steep Incline.
- delivery_address: The delivery address or coordinates (list).
- dropping_setup: Describe the dropping setup (similar to lifting setup). If unsure, say: "No problem, I understand. Let's continue and follow up later if needed."
- container_door_opening_drop_off: If applicable, ask: "Which way does the container door open at drop-off? Please choose one: [A - towards the truck cabin, B - Right side, C - behind the truck, D - Left side]. You may skip if unsure."
- drop_off_surface_type: What type of surface will the container be placed on upon delivery? (e.g., concrete, asphalt, grass, or dirt)
- drop_off_location_grade: What is the approximate grade of the drop-off location? If unsure, choose one: Flat Surface, Mild incline, or Steep Incline.

Keep your question clear, friendly, and natural.

Your response should only contain the question you want to ask.
        """
    )
)

# Less than 3 Containers - Updating State
state_update_prompt_less3 = PromptTemplate(
    input_variables=["history", "current_state", "question", "response"],
    template=(
        """
History:
{history}
Current container details:
{current_state}

The user responded to the question:
Question: {question}
Response: {response}

Please update the container details with the new information. Use the latest input if there’s any conflict.
If the user asked for clarifications, provide a friendly answer and refer to the conversation history if needed.

State fields (with guidelines):
- used_service_before: Have you used our service before? 
  - If Yes, respond with: "Before we proceed, I recommend reviewing our requirements page to ensure all containers meet our safety standards for transport. This includes restrictions on overhangs, protrusions, and proper corner castings. Do all your containers meet these criteria? If not, I’d be happy to guide you through our requirements. Please choose one of the two options."
  - If No, respond with: "I recommend reviewing our full requirements page to better understand our services, as we are not your typical container transport company. This should help clarify our offerings. Feel free to ask if you have any questions during the quote process."
- size: The size for each container (list).
- empty_or_loaded: The status of each container (list). if empty then make hazardous as No.
- hazardous: Any hazardous materials being transported (if yes, thank the user and ask if there’s anything else hazardous).
- new_customer: Information on whether you are a new customer and details about corner castings and any protrusions.
- pickup_address: The pickup address (list).
- lifting_setup: The lifting setup details (list).
- container_door_opening_pickup: The container door opening direction for pickup (list).
- pickup_surface_type: The surface type at pickup (list).
- pickup_location_grade: The pickup location grade (list).
- delivery_address: The delivery address or coordinates (list).
- dropping_setup: The dropping setup details (list).
- container_door_opening_drop_off: The container door opening direction at drop-off (list).
- drop_off_surface_type: The surface type at drop-off (list).
- drop_off_location_grade: The drop-off location grade (list).

Output Format:
{{
    "updated_state": {{
        "used_service_before": "...",
        "size": ["...", "..."],
        "empty_or_loaded": ["...", "..."],
        "hazardous": ["...", "..."],
        "new_customer": ["...", "..."],
        "pickup_address": ["...", "..."],
        "lifting_setup": ["...", "..."],
        "container_door_opening_pickup": ["...", "..."],
        "pickup_surface_type": ["...", "..."],
        "pickup_location_grade": ["...", "..."],
        "delivery_address": ["...", "..."],
        "dropping_setup": ["...", "..."],
        "container_door_opening_drop_off": ["...", "..."],
        "drop_off_surface_type": ["...", "..."],
        "drop_off_location_grade": ["...", "..."]
    }},
    "response": "Your response to the user here."
}}

Output only the JSON object.
        """
    )
)


# Global variable to hold the current question.
current_question = None

# --- Node Functions ---

def ask_question_node(state: ShippingWorkflowState) -> ShippingWorkflowState:
    if is_complete_personal_detail(state):
        return state

    # Retrieve session history and format as string.
    history = get_by_session_id(session_id)
    history_str = "\n".join(history)
    
    # Format the prompt with the current state and chat history.
    formatted_prompt = question_prompt_personal_detail.format(history=history_str, current_state=state["personal_detail"])
    
    # Call the LLM to generate a question.
    response = LLMChain(llm=llm, prompt=question_prompt_personal_detail).run({
        "history": history_str,
        "current_state": state["personal_detail"]
    })
    
    global current_question
    current_question = response
    
    # Append the question to the chat history.
    history.append("\nQuestion asked: " + response)
    return state

def process_answer_node(state: ShippingWorkflowState) -> ShippingWorkflowState:
    history = get_by_session_id(session_id)
    history_str = "\n".join(history)
    
    # Ask the user for their input in response to the current question.
    user_input = input(current_question + "\nYour answer: ")
    
    # Format the state update prompt.
    formatted_prompt = state_update_prompt_personal_detail.format(
        history=history_str, current_state=state["personal_detail"],
        question=current_question, response=user_input
    )
    
    # Call the LLM to update the state.
    response = LLMChain(llm=llm, prompt=state_update_prompt_personal_detail).run({
        "history": history_str,
        "current_state": state["personal_detail"],
        "question": current_question,
        "response": user_input
    })
    
    cleaned_res = clean_json_response(response)
    try:
        new_json = json.loads(cleaned_res)
        new_state = new_json["updated_state"]
        # clear the ouput before printing
        print("\033[H\033[J")
        print("Response from Chatbot: ", new_json["response"])
        
        
        state["personal_detail"].update(new_state)
        print("Updated State: ", state)
        if state["personal_detail"]["number_of_containers"] != None:
            state["more_3"]["number_of_containers"] = state["personal_detail"]["number_of_containers"]
            state["less_3"]["number_of_containers"] = state["personal_detail"]["number_of_containers"]
    except json.JSONDecodeError as e:
        print("Error decoding JSON:", e)
        print("Response:", cleaned_res)
    
    # Append the update to the chat history.
    history.append("User Input: "+user_input+"Response from Chatbot: "+new_json["response"]+"\nUpdated State: " + cleaned_res)
    return state

def more_than_3_ask_node(state: ShippingWorkflowState) -> ShippingWorkflowState:
    if is_complete_more3(state):
        return state

    # Retrieve session history and format as string.
    history = get_by_session_id(session_id)
    history_str = "\n".join(history)
    
    # Format the prompt with the current state and chat history.
    formatted_prompt = question_prompt_more3.format(history=history_str, current_state=state["more_3"])
    
    # Call the LLM to generate a question.
    response = LLMChain(llm=llm, prompt=question_prompt_more3).run({
        "history": history_str,
        "current_state": state["more_3"]
    })
    
    global current_question
    current_question = response
    
    # Append the question to the chat history.
    history.append("\nQuestion asked: " + response)
    return state

def more_than_3_process_node(state: ShippingWorkflowState) -> ShippingWorkflowState:
    history = get_by_session_id(session_id)
    history_str = "\n".join(history)
    
    # Ask the user for their input in response to the current question.
    user_input = input(current_question + "\nYour answer: ")
    
    # Format the state update prompt.
    formatted_prompt = state_update_prompt_more3.format(
        history=history_str, current_state=state["more_3"],
        question=current_question, response=user_input
    )
    
    # Call the LLM to update the state.
    response = LLMChain(llm=llm, prompt=state_update_prompt_more3).run({
        "history": history_str,
        "current_state": state["more_3"],
        "question": current_question,
        "response": user_input
    })
    
    cleaned_res = clean_json_response(response)
    try:
        new_json = json.loads(cleaned_res)
        new_state = new_json["updated_state"]
        # clear the ouput before printing
        print("\033[H\033[J")
        print("Response from Chatbot: ", new_json["response"])
        
        # keys_to_exclude = {'empty_or_loaded', 'pickup_address', 'delivery_address'}
        # state.update({k: v for k, v in new_state.items() if k not in keys_to_exclude})
        state["more_3"].update(new_state)
        print("Updated State: ", state)
    except json.JSONDecodeError as e:
        print("Error decoding JSON:", e)
        print("Response:", cleaned_res)
    
    # Append the update to the chat history.
    history.append("User Input: "+user_input+"Response from Chatbot: "+new_json["response"]+"\nUpdated State: " + cleaned_res)
    return state

def less_than_3_ask_node(state: ShippingWorkflowState) -> ShippingWorkflowState:
    if is_complete_less3(state):
        return state

    # Retrieve session history and format as string.
    history = get_by_session_id(session_id)
    history_str = "\n".join(history)
    
    # Format the prompt with the current state and chat history.
    formatted_prompt = question_prompt_less3.format(history=history_str, current_state=state["less_3"])
    
    # Call the LLM to generate a question.
    response = LLMChain(llm=llm, prompt=question_prompt_less3).run({
        "history": history_str,
        "current_state": state["less_3"]
    })
    
    global current_question
    current_question = response
    
    # Append the question to the chat history.
    history.append("\nQuestion asked: " + response)
    return state

def less_than_3_process_node(state: ShippingWorkflowState) -> ShippingWorkflowState:
    history = get_by_session_id(session_id)
    history_str = "\n".join(history)
    
    # Ask the user for their input in response to the current question.
    user_input = input(current_question + "\nYour answer: ")
    
    # Format the state update prompt.
    formatted_prompt = state_update_prompt_less3.format(
        history=history_str, current_state=state["less_3"],
        question=current_question, response=user_input
    )
    
    # Call the LLM to update the state.
    response = LLMChain(llm=llm, prompt=state_update_prompt_less3).run({
        "history": history_str,
        "current_state": state["less_3"],
        "question": current_question,
        "response": user_input
    })
    
    cleaned_res = clean_json_response(response)
    try:
        new_json = json.loads(cleaned_res)
        new_state = new_json["updated_state"]
        # clear the ouput before printing
        print("\033[H\033[J")
        print("Response from Chatbot: ", new_json["response"])
        
        # keys_to_exclude = {'empty_or_loaded', 'pickup_address', 'delivery_address'}
        # state.update({k: v for k, v in new_state.items() if k not in keys_to_exclude})
        state["less_3"].update(new_state)
        print("Updated State: ", state)
    except json.JSONDecodeError as e:
        print("Error decoding JSON:", e)
        print("Response:", cleaned_res)
    
    # Append the update to the chat history.
    history.append("User Input: "+user_input+"Response from Chatbot: "+new_json["response"]+"\nUpdated State: " + cleaned_res)
    return state

def workflow_complete_node(state: ShippingWorkflowState) -> ShippingWorkflowState:
    print("Workflow completed.")
    print("Final state:", state)
    return state

# --- Conditional Routing Functions ---
def route_after_ask_personal_detail(state: ShippingWorkflowState) -> str:
    # print("in route_after_ask_personal_detail")
    if is_complete_personal_detail(state):
        if int(state["personal_detail"]["number_of_containers"]) >= 3:
            return "more_than_3_ask"
        else:
            return "less_than_3_ask"
    else:
        return "process_answer"
    

def route_after_process_personal_detail(state: ShippingWorkflowState) -> str:
    # print("in route_after_process_personal_detail")
    if is_complete_personal_detail(state):
        if int(state["personal_detail"]["number_of_containers"]) >= 3:
            return "more_than_3_ask"
        else:
            return "less_than_3_ask"
    else:
        return "ask_question"

def route_after_ask_more3(state: ShippingWorkflowState) -> str:
    return "workflow_complete" if is_complete_more3(state) else "more_than_3_process"

def route_after_process_more3(state: ShippingWorkflowState) -> str:
    return "workflow_complete" if is_complete_more3(state) else "more_than_3_ask"

def route_after_ask_less3(state: ShippingWorkflowState) -> str:
    return "workflow_complete" if is_complete_less3(state) else "less_than_3_process"

def route_after_process_less3(state: ShippingWorkflowState) -> str:
    return "workflow_complete" if is_complete_less3(state) else "less_than_3_ask"


# --- Build the LangGraph State Graph ---
workflow = StateGraph(ShippingWorkflowState)
workflow.add_node("ask_question", ask_question_node)
workflow.add_node("process_answer", process_answer_node)
workflow.add_node("more_than_3_ask", more_than_3_ask_node)
workflow.add_node("more_than_3_process", more_than_3_process_node)
workflow.add_node("less_than_3_ask", less_than_3_ask_node)
workflow.add_node("less_than_3_process", less_than_3_process_node)

workflow.add_node("workflow_complete", workflow_complete_node)

workflow.add_edge(START, "ask_question")
workflow.add_conditional_edges(
    "ask_question",
    route_after_ask_personal_detail,
    {"workflow_complete": "workflow_complete", "process_answer": "process_answer", "more_than_3_ask": "more_than_3_ask", "less_than_3_ask": "less_than_3_ask"}
)
workflow.add_conditional_edges(
    "process_answer",
    route_after_process_personal_detail,
    {"workflow_complete": "workflow_complete", "ask_question": "ask_question", "more_than_3_ask": "more_than_3_ask", "less_than_3_ask": "less_than_3_ask"}
)

workflow.add_conditional_edges(
    "more_than_3_ask",
    route_after_ask_more3,
    {"workflow_complete": "workflow_complete", "more_than_3_process": "more_than_3_process"}
)

workflow.add_conditional_edges(
    "more_than_3_process",
    route_after_process_more3,
    {"workflow_complete": "workflow_complete", "more_than_3_ask": "more_than_3_ask"}
)

workflow.add_conditional_edges(
    "less_than_3_ask",
    route_after_ask_less3,
    {"workflow_complete": "workflow_complete", "less_than_3_process": "less_than_3_process"}
)

workflow.add_conditional_edges(
    "less_than_3_process",
    route_after_process_less3,
    {"workflow_complete": "workflow_complete", "less_than_3_ask": "less_than_3_ask"}
)



workflow.add_edge("workflow_complete", END)

app = workflow.compile()

# --- Main Loop ---
if __name__ == "__main__":
    # Initialize state with all fields empty.
    personal_detail: Optional[dict] = {
    "full_name": None,
    "company_name": None,
    "company_address": None,
    "phone_number": None,
    "email": None,
    "number_of_containers": None
    }

    more_3: Optional[dict] = {
        "number_of_containers": None,
        "size": None,  # Size of the containers
        "empty_or_loaded": None,  # List of container statuses
        "pickup_address": None,   # Pickup addresses for the containers
        "delivery_address": None  # Delivery addresses for the containers
    }
    
    less_3: Optional[dict] = {
        "number_of_containers": None,
        "used_service_before": None,
        "size": None,
        "empty_or_loaded": None,
        "hazardous": None,
        "new_customer": None,
        "pickup_address": None,
        "lifting_setup": None,
        "container_door_opening_pickup": None,
        "pickup_surface_type": None,
        "pickup_location_grade": None,
        "delivery_address": None,
        "dropping_setup": None,
        "container_door_opening_drop_off": None,
        "drop_off_surface_type": None,
        "drop_off_location_grade": None
        
    }


    # The final state will now have these steps as part of it.
    state: ShippingWorkflowState = {
        "personal_detail": personal_detail,
        "more_3": more_3,
        "less_3": less_3
    }
    print("Initial state:", state)
    state = app.invoke(state,{"recursion_limit": 100})
    print("Final state:", state)
    print("Chat history store:", store)


Initial state: {'personal_detail': {'full_name': None, 'company_name': None, 'company_address': None, 'phone_number': None, 'email': None, 'number_of_containers': None}, 'more_3': {'number_of_containers': None, 'size': None, 'empty_or_loaded': None, 'pickup_address': None, 'delivery_address': None}, 'less_3': {'number_of_containers': None, 'used_service_before': None, 'size': None, 'empty_or_loaded': None, 'hazardous': None, 'new_customer': None, 'pickup_address': None, 'lifting_setup': None, 'container_door_opening_pickup': None, 'pickup_surface_type': None, 'pickup_location_grade': None, 'delivery_address': None, 'dropping_setup': None, 'container_door_opening_drop_off': None, 'drop_off_surface_type': None, 'drop_off_location_grade': None}}

Response from Chatbot:  Thank you for providing your details, John Doe. If you have any more questions or need further assistance, feel free to ask!
Updated State:  {'personal_detail': {'full_name': 'John Doe', 'company_name': 'ABC Inc.', 'compan

In [ ]:

test_inputs = [
    "HI I am John Doe from ABC Inc. My address is 123 Main St, Dallas, TX 75201. Phone: 555-123-4567, Email: jhondoe@gmail.com",
    "We have 4 containers to  ship",
    "The container is 40ft and loaded. Pickup address is 456 Elm St, Dallas, TX 75202",
    "The delivery address is 456 Elm St, Dallas, TX 75202",
    "We have 2 containers to ship",
    "We have used the service before",
    "The container is 20ft and loaded. Pickup address is 456 Elm St, Dallas, TX 75202"
]

In [ ]:

# Prompt template to ask the user for missing information.
question_prompt_less3 = PromptTemplate(
    input_variables=["history", "current_state"],
    template=(
        """
        History:
        {history}
        This is my current data from the user:
        {current_state}
        1. The user has provided some details. Update the state if needed.
        2. Please ask for the missing information in the order of the fields.
        3. Keep the history of the chat in mind when asking the next question.
        4. The conversational can deviate from the expected flow. Be prepared to handle that.
        5. After a few interactions if the conversation is deviating, try to bring it back on track but in a way that feels natural.
        6. The comments in the () are the responses to the user's input that the chatbot should give.
        State description:
        [
            used_service_before: [Yes/No. If Yes, tell the user this,(Before we proceed, I recommend reviewing our requirements page to ensure all containers meet our safety standards for transport. This includes restrictions on overhangs or protrusions on any side of the container, as well as correct corner castings. Do all your containers meet these criteria? If not, I’d be happy to guide you through our requirements. Please note that not meeting these conditions may result in additional fees. Please select on of the two options.) If NO, tell the user this (I recommend reviewing our full requirements page to better understand our services, as we are not your typical container transport company. This should help clarify what we specifically offer. However, should you have any questions during the quote request process, I am here to guide you and provide answers, so please don't hesitate to ask.)]
            size: [Size of the containers. If number_of_containers = 2, this will be a list of 2 elements]
            empty_or_loaded: [Empty or loaded status of the containers. If number_of_containers = 2, this will be a list of 2 elements]
            hazardous: [Are you transporting any hazardous or combustible materials, such as propane, paint, or other fluids? If there are none, please respond with 'No.' If yes, could you please specify what you are transporting?. If number_of_containers = 2, this will be a list of 2 elements. If the user selects Yes, tell the user this (Thank you for letting me know about the propane tanks. I’ll make a note of that. Is there anything else in your shipment that could be hazardous?)]
            new_customer: [Are you a new customer? If Yes, Does your container have these universal 5/8 inch corner castings in good condition without major dints or defects?(Thank you for letting me know. Could you provide more details on the condition of the corner castings?). If number_of_containers = 2, this will be a list of 2 elements. Does your container have any protrusions on the ends or top, such as air conditioning units, brackets, metal signage, or electrical boxes?(Thank you for letting me know. Since there’s a protrusion on the top, we may not be able to transport it due to safety reasons. However, this is often case-specific. If possible, please send photos so our team can verify. You can do this later during the email process, so no worries if you don’t have the photos on hand right now.). Great, do either of the long sides of your container have any protrusions, such as air conditioning units, brackets, metal signage, or electrical boxes? If so, please describe what they are.(If Yes, (Thank you for letting me know. Since there are protrusions on the sides, we may not be able to transport it due to safety reasons. However, this is often case-specific. If possible, please send photos so our team can verify. You can do this later during the email process, so no worries if you don’t have the photos on hand right now.), If No, (Is there any additional information we should know about the container itself or its contents?))). If number_of_containers = 2, this will be a list of 2 elements.]
            pickup_address: [Pickup address for the containers. If number_of_containers = 2, this will be a list of 2 elements]
            lifting_setup: [Right/left side load/unload and 20, 40 or 60 feet.if unsure or neither(No problem, I completely understand. These diagrams are general guidelines, and we know that not all clients have the same setup. For now, let’s proceed, and a staff member may reach out for further details if needed. If possible, please send photos so our team can verify. You can do this later via email, so no worries if you don’t have them on hand right now.). And if this scenario haapens again (No problem, I completely understand. Let's go ahead and continue without this.). If number_of_containers = 2, this will be a list of 2 elements.]
            container_door_opening_pickup: [If lifting setup is right/left side load/unload and 20, 40 or 60 feet, then ask this question, which way does the container door open? Please choose one between [A- towards the truck cabin, B- Right side of container, C- behind the truck, D- Left side of the container] . If unsure or neither, skip this question. If number_of_containers = 2, this will be a list of 2 elements]
            pickup_surface_type: [Could you specify the type of surface the container will be pickup from? For example, will it be concrete, asphalt, grass, or dirt? If number_of_containers = 2, this will be a list of 2 elements]
            pickup_location_grade: [Do you know the approximate grade of the pickup location? It's important for ensuring we can feasibly park and pick up the container. If you're unsure, please choose one of the three options— Flat Surface, Mild incline or Steep Incline —that most closely resembles the area. If number_of_containers = 2, this will be a list of 2 elements]
            delivery_address: [Delivery address or Coordinates for the containers. If number_of_containers = 2, this will be a list of 2 elements]
            dropping_setup: [Right/left side load/unload and 20, 40 or 60 feet.if unsure or neither(No problem, I completely understand. These diagrams are general guidelines, and we know that not all clients have the same setup. For now, let’s proceed, and a staff member may reach out for further details if needed. If possible, please send photos so our team can verify. You can do this later via email, so no worries if you don’t have them on hand right now.). And if this scenario haapens again (No problem, I completely understand. Let's go ahead and continue without this.). If number_of_containers = 2, this will be a list of 2 elements.] 
            container_door_opening_drop_off: [If lifting setup is right/left side load/unload and 20, 40 or 60 feet, then ask this question, which way does the container door open? Please choose one between [A- towards the truck cabin, B- Right side of container, C- behind the truck, D- Left side of the container] . If unsure or neither, skip this question. If number_of_containers = 2, this will be a list of 2 elements]
            drop_off_surface_type: [Could you specify the type of surface the container will be placed on upon delivery? For example, will it be concrete, asphalt, grass, or dirt? If number_of_containers = 2, this will be a list of 2 elements]
            drop_off_location_grade: [Do you know the approximate grade of the drop off location? It's important for ensuring we can feasibly park and drop off the container. If you're unsure, please choose one of the three options— Flat Surface, Mild incline or Steep Incline —that most closely resembles the area. If number_of_containers = 2, this will be a list of 2 elements]
        ]
        
        Your response should only contain the question you want to ask.
        """
    )
)

# Prompt template to update the state based on the user's response.
state_update_prompt_less3 = PromptTemplate(
    input_variables=["history", "current_state", "question", "response"],
    template=(
        """
        History:
        {history}
        This is my current data from the user:
        {current_state}

        The user provided the following response to the question:
        Question: {question}
        Response: {response}

        Your tasks are:
        1. Update the state with the new information provided. If there is conflicting info, prioritize the new input.
        2. If the user asked a question or needs clarification, provide a friendly answer.
        3. User may ask some questions which have their answers in the history. You can use the history to answer those questions.
        4. The comments in the () are the responses to the user's input that the chatbot should give.
        State description:
        [
            used_service_before: [Yes/No. If Yes, tell the user this,(Before we proceed, I recommend reviewing our requirements page to ensure all containers meet our safety standards for transport. This includes restrictions on overhangs or protrusions on any side of the container, as well as correct corner castings. Do all your containers meet these criteria? If not, I’d be happy to guide you through our requirements. Please note that not meeting these conditions may result in additional fees. Please select on of the two options.) If NO, tell the user this (I recommend reviewing our full requirements page to better understand our services, as we are not your typical container transport company. This should help clarify what we specifically offer. However, should you have any questions during the quote request process, I am here to guide you and provide answers, so please don't hesitate to ask.)]
            size: [Size of the containers. If number_of_containers = 2, this will be a list of 2 elements]
            empty_or_loaded: [Empty or loaded status of the containers. If number_of_containers = 2, this will be a list of 2 elements]
            hazardous: [Are you transporting any hazardous or combustible materials, such as propane, paint, or other fluids? If there are none, please respond with 'No.' If yes, could you please specify what you are transporting?. If number_of_containers = 2, this will be a list of 2 elements. If the user selects Yes, tell the user this (Thank you for letting me know about the propane tanks. I’ll make a note of that. Is there anything else in your shipment that could be hazardous?)]
            new_customer: [Are you a new customer? If Yes, Does your container have these universal 5/8 inch corner castings in good condition without major dints or defects?(Thank you for letting me know. Could you provide more details on the condition of the corner castings?). If number_of_containers = 2, this will be a list of 2 elements. Does your container have any protrusions on the ends or top, such as air conditioning units, brackets, metal signage, or electrical boxes?(Thank you for letting me know. Since there’s a protrusion on the top, we may not be able to transport it due to safety reasons. However, this is often case-specific. If possible, please send photos so our team can verify. You can do this later during the email process, so no worries if you don’t have the photos on hand right now.). Great, do either of the long sides of your container have any protrusions, such as air conditioning units, brackets, metal signage, or electrical boxes? If so, please describe what they are.(If Yes, (Thank you for letting me know. Since there are protrusions on the sides, we may not be able to transport it due to safety reasons. However, this is often case-specific. If possible, please send photos so our team can verify. You can do this later during the email process, so no worries if you don’t have the photos on hand right now.), If No, (Is there any additional information we should know about the container itself or its contents?))). If number_of_containers = 2, this will be a list of 2 elements.]
            pickup_address: [Pickup address for the containers. If number_of_containers = 2, this will be a list of 2 elements]
            lifting_setup: [Right/left side load/unload and 20, 40 or 60 feet.if unsure or neither(No problem, I completely understand. These diagrams are general guidelines, and we know that not all clients have the same setup. For now, let’s proceed, and a staff member may reach out for further details if needed. If possible, please send photos so our team can verify. You can do this later via email, so no worries if you don’t have them on hand right now.). And if this scenario haapens again (No problem, I completely understand. Let's go ahead and continue without this.). If number_of_containers = 2, this will be a list of 2 elements.]
            container_door_opening_pickup: [If lifting setup is right/left side load/unload and 20, 40 or 60 feet, then ask this question, which way does the container door open? Please choose one between [A- towards the truck cabin, B- Right side of container, C- behind the truck, D- Left side of the container] . If unsure or neither, skip this question. If number_of_containers = 2, this will be a list of 2 elements]
            pickup_surface_type: [Could you specify the type of surface the container will be pickup from? For example, will it be concrete, asphalt, grass, or dirt? If number_of_containers = 2, this will be a list of 2 elements]
            pickup_location_grade: [Do you know the approximate grade of the pickup location? It's important for ensuring we can feasibly park and pick up the container. If you're unsure, please choose one of the three options— Flat Surface, Mild incline or Steep Incline —that most closely resembles the area. If number_of_containers = 2, this will be a list of 2 elements]
            delivery_address: [Delivery address or Coordinates for the containers. If number_of_containers = 2, this will be a list of 2 elements]
            dropping_setup: [Right/left side load/unload and 20, 40 or 60 feet.if unsure or neither(No problem, I completely understand. These diagrams are general guidelines, and we know that not all clients have the same setup. For now, let’s proceed, and a staff member may reach out for further details if needed. If possible, please send photos so our team can verify. You can do this later via email, so no worries if you don’t have them on hand right now.). And if this scenario haapens again (No problem, I completely understand. Let's go ahead and continue without this.). If number_of_containers = 2, this will be a list of 2 elements.] 
            container_door_opening_drop_off: [If lifting setup is right/left side load/unload and 20, 40 or 60 feet, then ask this question, which way does the container door open? Please choose one between [A- towards the truck cabin, B- Right side of container, C- behind the truck, D- Left side of the container] . If unsure or neither, skip this question. If number_of_containers = 2, this will be a list of 2 elements]
            drop_off_surface_type: [Could you specify the type of surface the container will be placed on upon delivery? For example, will it be concrete, asphalt, grass, or dirt? If number_of_containers = 2, this will be a list of 2 elements]
            drop_off_location_grade: [Do you know the approximate grade of the drop off location? It's important for ensuring we can feasibly park and drop off the container. If you're unsure, please choose one of the three options— Flat Surface, Mild incline or Steep Incline —that most closely resembles the area. If number_of_containers = 2, this will be a list of 2 elements]
        ]

        Output Format:
        {{
            "updated_state": {{
                "used_service_before": "...",
                "size": ["...", "..."],
                "empty_or_loaded": ["...", "..."],
                "hazardous": ["...", "..."],
                "new_customer": ["...", "..."],
                "pickup_address": ["...", "..."],
                "lifting_setup": ["...", "..."],
                "container_door_opening_pickup": ["...", "..."],
                "pickup_surface_type": ["...", "..."],
                "pickup_location_grade": ["...", "..."],
                "delivery_address": ["...", "..."],
                "dropping_setup": ["...", "..."],
                "container_door_opening_drop_off": ["...", "..."],
                "drop_off_surface_type": ["...", "..."],
                "drop_off_location_grade": ["...", "..."]
            }},
            "response": "Your response to the user here."
        }}

        Output only the JSON object.
        """
    )
)